# CodaMOSA vs CoverUp — Experiment Control

This notebook is the single observable control panel. It calls versioned scripts; it does not duplicate the experiment logic.

**Design interpretation:** File B wins. This repository runs CodaMOSA, CoverUp, and the Pynguin baseline on MT and CM. B1 = gpt-5.6-sol, B2 = glm-5.3-flash (router.octalabs.id/v1). MuTAP rows come from the collaborator.

**Safety:** edit `config/local.env`, which is gitignored. Never paste keys into notebook cells or `experiment.toml`.

**Kernel:** use `.venv (Python 3.14.4)`. This notebook must be opened from inside the repository directory.


## 0. Kernel and local credentials

Use `.venv (Python 3.14.4)`. Fill `config/local.env` outside this notebook. This cell prints presence only, never values.


In [1]:
from pathlib import Path
import sys

# Find the repo root by walking up from the current directory, and fail loudly
# if it cannot be found instead of silently falling back to "/".
_root = Path.cwd().resolve()
while not (_root / "config" / "experiment.toml").exists():
    if _root == _root.parent:
        raise RuntimeError(
            "config/experiment.toml not found above the current directory "
            f"({_root}). Open this notebook from inside the repository "
            "directory and restart the kernel before running this cell."
        )
    _root = _root.parent
ROOT = _root

_scripts = str(ROOT / "scripts")
if _scripts in sys.path:
    sys.path.remove(_scripts)
sys.path.insert(0, _scripts)

from notebook_control import load_local_env, presence, run, print_status

loaded = load_local_env()
print("Loaded variable names:", loaded)
print("Presence:", presence())
print("ROOT:", ROOT)
assert all(presence().values()), "Fill config/local.env before API probes."


Loaded variable names: ['LLM_BASE_URL', 'LLM_API_KEY', 'LLM_BASE_URL_B2', 'LLM_API_KEY_B2']
Presence: {'LLM_API_KEY': True, 'LLM_BASE_URL': True, 'LLM_API_KEY_B2': True, 'LLM_BASE_URL_B2': True}
ROOT: /home/masoyi/Teori-dalam-pengujian-perangkat-lunak


## 1. No-cost setup and validation

Creates/updates the environment, inventories 588 tasks, builds the 80-run pilot matrix (incl. Pynguin baseline rows), and runs tests. Safe to rerun.


In [2]:
run(["bash", "scripts/setup.sh"], timeout=1200)

$ bash scripts/setup.sh
Checked 12 packages in 49ms
Resolved 98 packages in 1.20s
   Building coverup @ file:///home/masoyi/Teori-dalam-pengujian-perangkat-lunak/coverup
      Built coverup @ file:///home/masoyi/Teori-dalam-pengujian-perangkat-lunak/coverup
Prepared 1 package in 255ms
Uninstalled 1 package in 2ms
Installed 1 package in 1ms
 ~ coverup==0.6.3 (from file:///home/masoyi/Teori-dalam-pengujian-perangkat-lunak/coverup)
Installed kernelspec teori-experiment in /home/masoyi/.local/share/jupyter/kernels/teori-experiment
MT tasks: 163
CM tasks: 425
Total available tasks: 588
Wrote 80 runs to results/pilot_matrix.csv
.........                                                                [100%]
9 passed in 1.99s

exit=0 elapsed=5.7s


CompletedProcess(args=['bash', 'scripts/setup.sh'], returncode=0, stdout='\x1bChecked \x1b12 packages\x1b \x1bin 49ms\x1b\x1b\n\x1bResolved \x1b98 packages\x1b \x1bin 1.20s\x1b\x1b\n   \x1b\x1bBuilding\x1b\x1b coverup\x1b @ file:///home/masoyi/Teori-dalam-pengujian-perangkat-lunak/coverup\x1b\n      \x1b\x1bBuilt\x1b\x1b coverup\x1b @ file:///home/masoyi/Teori-dalam-pengujian-perangkat-lunak/coverup\x1b\n\x1bPrepared \x1b1 package\x1b \x1bin 255ms\x1b\x1b\n\x1bUninstalled \x1b1 package\x1b \x1bin 2ms\x1b\x1b\n\x1bInstalled \x1b1 package\x1b \x1bin 1ms\x1b\x1b\n \x1b~\x1b \x1bcoverup\x1b\x1b==0.6.3 (from file:///home/masoyi/Teori-dalam-pengujian-perangkat-lunak/coverup)\x1b\nInstalled kernelspec teori-experiment in /home/masoyi/.local/share/jupyter/kernels/teori-experiment\nMT tasks: 163\nCM tasks: 425\nTotal available tasks: 588\nWrote 80 runs to results/pilot_matrix.csv\n\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b                              

## 2. Inspect dataset inventory and pilot plan

Expected totals: 163 MT functions + 425 CM modules = 588. Pilot: 5 MT + 3 disjoint CM × 2 tools × 2 backbones × R=2 = 64 runs.


In [3]:
run([sys.executable, "scripts/inventory.py"])
run([sys.executable, "scripts/build_matrix.py", "--pilot"])
run([sys.executable, "scripts/run_experiment.py", "--pilot", "--dry-run"])
print_status()

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/.venv/bin/python scripts/inventory.py
MT tasks: 163
CM tasks: 425
Total available tasks: 588

exit=0 elapsed=0.9s
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/.venv/bin/python scripts/build_matrix.py --pilot
Wrote 80 runs to results/pilot_matrix.csv

exit=0 elapsed=0.1s
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/.venv/bin/python scripts/run_experiment.py --pilot --dry-run
{
  "matrix": "/home/masoyi/Teori-dalam-pengujian-perangkat-lunak/results/pilot_matrix.csv",
  "results": "/home/masoyi/Teori-dalam-pengujian-perangkat-lunak/results/pilot_results.csv",
  "total": 80,
  "completed_ok": 64,
  "recorded": 64,
  "pending": 16,
  "cost_usd": 8.774848
}

exit=0 elapsed=0.1s
{
  "credentials": {
    "LLM_API_KEY": true,
    "LLM_BASE_URL": true,
    "LLM_API_KEY_B2": true,
    "LLM_BASE_URL_B2": true
  },
  "files": {
    "pilot_matrix.csv": {
      "rows": 80,
      "ok": 0,
      "failed": 0,
      "pending": 80,
     

In [4]:
run([sys.executable, "scripts/report.py"])
from IPython.display import Markdown
Markdown((ROOT / "experiment_report.md").read_text(encoding="utf-8"))

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/.venv/bin/python scripts/report.py
Wrote Markdown report to /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/experiment_report.md

exit=0 elapsed=0.1s


# Laporan Eksperimen CodaMOSA vs CoverUp

## 1. Status

Dokumen ini dibuat otomatis dari artefak pipeline. Angka hasil hanya diisi dari data yang benar-benar diukur; angka dummy File A tidak digunakan sebagai target atau hasil.

- Probe endpoint: **selesai**
- Pilot R=2: **selesai/sebagian**
- Run utama R=10: **selesai/sebagian**

## 2. Interpretasi Rancangan

File B bersifat otoritatif. Pipeline ini mengukur CodaMOSA dan CoverUp pada MT dan CM dengan dua backbone. Hasil Pynguin dan MuTAP akan digabung dari kolaborator menggunakan skema hasil yang sama.

Perbandingan tool pada suite yang sama bersifat berpasangan per unit. Perbandingan MT dengan CM untuk tool yang sama bersifat deskriptif karena unit MT dan CM tidak berpasangan.

## 3. Inventaris Dataset

| Suite | Definisi satu task | Tersedia | Dipakai pilot | Dipakai run utama |
|---|---|---|---|---|
| MT | Satu fungsi benchmark dalam satu modul f001–f163 | 163 | 5 | 163 |
| CM | Satu modul Python pada satu baris cm_modules.csv | 425 | 3 | 40 |
| Total | — | 588 | 8 | 203 |

Inventaris mentah berisi **163 MT + 425 CM = 588 task**. Sesuai File B, run utama tidak memakai seluruh CM; ia memakai seluruh 163 MT dan sampel CM bertingkat sebanyak 40, sehingga terdapat **203 unit analisis utama**.

### Sampel CM per proyek

| Proyek | Jumlah |
|---|---|
| PySnooper | 1 |
| ansible | 10 |
| black | 1 |
| dataclasses-json | 1 |
| flutils | 1 |
| httpie | 3 |
| isort | 1 |
| py-backwards | 2 |
| pyMonet | 1 |
| pypara | 1 |
| python-semantic-release | 1 |
| pytutils | 1 |
| sty | 1 |
| tornado | 1 |
| typesystem | 4 |
| youtube-dl | 10 |

### Sampel CM per stratum

| Stratum | Jumlah |
|---|---|
| L0B0 | 6 |
| L0B1 | 6 |
| L1B0 | 5 |
| L1B1 | 5 |
| L1B2 | 4 |
| L2B0 | 4 |
| L2B1 | 5 |
| L2B2 | 5 |

Deviasi yang disetujui: stratum L0B2 kosong pada populasi aktual, sehingga sampel mencakup seluruh delapan stratum non-kosong, bukan memaksakan stratum kesembilan.

## 4. Matriks Run

| Tahap | Jumlah run |
|---|---|
| Pilot: 5 MT + 3 CM × 2 tool × 2 backbone × R=2 | 80 |
| Utama: 163 MT + 40 CM × 2 tool × 2 backbone × R=10 | 10150 |

## 5. Model dan Probe

| Backbone | Requested | Returned | Function calling | Status/Error |
|---|---|---|---|---|
| B1 | gpt-5.6-sol | gpt-5.6-sol | True | PASS |
| B2 | glm-5.3-flash | glm-5.3-flash | True | PASS |

B1 dilaporkan sebagai `gpt-5.6-sol`; model terverifikasi dari respons endpoint (nama model spesifik dikembalikan sejak router per 2026-10-27). Run historis (22–26 Okt 2026) memakai router `auto` yang identitas per-panggilannya tidak diungkap; nilai asli tersimpan di `results/raw/*/llm_log.jsonl`.

## 6. Hasil Pilot

| Suite | Tool | Backbone | Run tercatat | OK | Gagal | Biaya USD |
|---|---|---|---|---|---|---|
| CM | CodaMOSA | B1 | 6 | 6 | 0 | 2.913744 |
| CM | CodaMOSA | B2 | 6 | 6 | 0 | 3.129988 |
| CM | CoverUp | B1 | 6 | 6 | 0 | 0.402708 |
| CM | CoverUp | B2 | 6 | 6 | 0 | 0.331296 |
| MT | CodaMOSA | B1 | 10 | 10 | 0 | 0.452280 |
| MT | CodaMOSA | B2 | 10 | 10 | 0 | 0.685280 |
| MT | CoverUp | B1 | 10 | 10 | 0 | 0.428944 |
| MT | CoverUp | B2 | 10 | 10 | 0 | 0.430608 |

## 7. Hasil Utama

| Suite | Tool | Backbone | Run tercatat | OK | Gagal | Biaya USD |
|---|---|---|---|---|---|---|
| CM | CodaMOSA | B1 | 6 | 6 | 0 | 2.913744 |
| CM | CodaMOSA | B2 | 6 | 6 | 0 | 3.129988 |
| CM | CoverUp | B1 | 6 | 6 | 0 | 0.402708 |
| CM | CoverUp | B2 | 6 | 6 | 0 | 0.331296 |
| CM | Pynguin |  | 357 | 234 | 123 | 0.000000 |
| MT | CodaMOSA | B1 | 10 | 10 | 0 | 0.452280 |
| MT | CodaMOSA | B2 | 10 | 10 | 0 | 0.685280 |
| MT | CoverUp | B1 | 10 | 10 | 0 | 0.428944 |
| MT | CoverUp | B2 | 10 | 10 | 0 | 0.430608 |
| MT | MuTAP | B1 | 1533 | 1191 | 342 | 43.557836 |
| MT | Pynguin |  | 1449 | 1412 | 37 | 0.000000 |

Analisis akhir menggunakan mean, median, dan standard deviation; Wilcoxon signed-rank berpasangan; koreksi Holm; Vargha–Delaney A12; dan bootstrap CI 95% untuk selisih mean.

## 8. Task Gagal atau Dikeluarkan

Daftar rinci tersedia di `results/analysis/excluded_tasks.csv` atau `results/pilot_analysis/excluded_tasks.csv`. Tidak ada task yang boleh dikeluarkan untuk menyesuaikan hasil dengan File A.

## 9. Deviasi dari File B

1. File A memuat angka ilustratif bertanda dummy; angka tersebut tidak dipakai sebagai ekspektasi.
2. File B merevisi RQ3 menjadi cost-effectiveness; revisi ini mengalahkan rancangan korelasi karakteristik kode pada File A.
3. Pipeline lokal menjalankan CodaMOSA dan CoverUp; Pynguin dan MuTAP ditangani kolaborator.
4. Populasi CM hanya memiliki delapan stratum tertil non-kosong; penggunaan delapan stratum telah disetujui.
5. B2 sementara menggunakan `auto` yang sama sebagai B1 karena endpoint konfigurasi B2 hanya mengekspos `auto`. Deviasi ini dicatat di docs/protocol.md dan tidak memenuhi dua rambang File B.

## 10. Keterbatasan

- Identitas GPT-5.6 Sol pada endpoint `auto` adalah klaim penjual dan tidak diverifikasi.
- Perbandingan lintas dataset tidak berpasangan karena unit MT dan CM berbeda.
- Hasil final tidak boleh disimpulkan sebelum probe B2, pilot, sanity checks, dan run utama selesai.
- Efek kontaminasi data tetap mungkin karena benchmark bersifat publik.

## 11. Artefak Reproduksi

- `config/experiment.toml`: protokol mesin-baca (tanpa secrets)
- `experiment_control.ipynb`: kontrol dan observasi proses
- `scripts/setup.sh`: setup tunggal
- `scripts/run_experiment.py`: entry point tunggal dan resumable
- `results/module_characteristics.csv`, `results/cm_sample.csv`, `results/*matrix.csv`
- `results/raw/<run_id>/`: request/response teredaksi
- `results/generated/<run_id>/`: generated tests dan logs
- `results/analysis/`: tabel statistik, failure rate, task exclusion, dan deviations


## 3. Endpoint/tool probes — required before pilot

This validates B1/B2 `model_returned`, CoverUp function calling, and the per-call rate gate. The probe sends `tool_choice: required` (string form) because this endpoint rejects the object form. B1 = `gpt-5.6-sol`, B2 = `glm-5.3-flash` on router.octalabs.id/v1 (File B two-backbone design).


In [5]:
probe_run = run([sys.executable, "scripts/probe_endpoints.py"], timeout=300, check=False)
probe = ROOT / "results" / "endpoint_probe.json"
probe_results = __import__("json").loads(probe.read_text(encoding="utf-8"))
for item in probe_results:
    print(item)
failed = [item for item in probe_results if not item.get("passed")]
PROBES_PASSED = not failed
if failed:
    print("\nBLOCKED: a backbone probe did not pass. Fix the failing backbone, rerun this cell, then continue to Section 4.")
    print("Probe report saved to results/endpoint_probe.json")
else:
    print("Both endpoint function-calling probes passed. Continue to pilot smoke test.")
PROBES_PASSED

$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/.venv/bin/python scripts/probe_endpoints.py
{
  "backbone": "B1",
  "passed": true,
  "model_requested": "gpt-5.6-sol",
  "model_returned": "gpt-5.6-sol",
  "function_calling": true,
  "http_status": 200,
  "tokens_in": 52,
  "tokens_out": 1,
  "finish_reason": "tool_calls"
}
{
  "backbone": "B2",
  "passed": true,
  "model_requested": "glm-5.3-flash",
  "model_returned": "glm-5.3-flash",
  "function_calling": true,
  "http_status": 200,
  "tokens_in": 166,
  "tokens_out": 40,
  "finish_reason": "tool_calls"
}
Both endpoint function-calling probes passed; wrote /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/results/endpoint_probe.json

exit=0 elapsed=8.7s
{'backbone': 'B1', 'passed': True, 'model_requested': 'gpt-5.6-sol', 'model_returned': 'gpt-5.6-sol', 'function_calling': True, 'http_status': 200, 'tokens_in': 52, 'tokens_out': 1, 'finish_reason': 'tool_calls'}
{'backbone': 'B2', 'passed': True, 'model_requested': 'glm-5.3-fla

True

## 4. Pilot — do not run until Section 3 passes

Runs 80 isolated jobs with checkpoints, 600 s budgets, configured workers, raw logs, and the USD 5,000 cost gates. This cell auto-skips when the pilot is already complete.

**Pause / resume:** The runner checkpoints every completed run to `results/pilot_results.csv`. To pause: `touch results/pause.requested` (or run `python scripts/run_experiment.py --pause`). The current in-flight run will finish, then no new runs launch. To resume: `python scripts/run_experiment.py --resume --retry-failed --pilot`. Re-running without `--retry-failed` skips already-completed runs automatically.


In [6]:
PROBES_PASSED = globals().get("PROBES_PASSED", False)
CONFIRM_PAID_PILOT = True

import csv as _csv
_pilot = ROOT / "results" / "pilot_results.csv"
_pilot_rows = list(_csv.DictReader(_pilot.open(newline="", encoding="utf-8"))) if _pilot.exists() else []
_pilot_ok = sum(1 for r in _pilot_rows if r.get("status") == "ok")

if _pilot_rows and _pilot_ok >= 0.8 * len(_pilot_rows):
    print(f"SKIPPED: pilot already complete ({_pilot_ok}/{len(_pilot_rows)} OK). "
          "Use the dual-runner cell below for the main queues.")
elif not PROBES_PASSED:
    print("SKIPPED: Section 3 probe gate has not passed. No pilot task was started.")
elif not CONFIRM_PAID_PILOT:
    print("SKIPPED: set CONFIRM_PAID_PILOT=True after reviewing probe results.")
else:
    try:
        run([sys.executable, "scripts/run_experiment.py", "--pilot", "--retry-failed", "--max-runs", "1", "--workers", "1"], timeout=2400)
        print_status()
    except RuntimeError as exc:
        print(f"\nSmoke run failed: {exc}")
        print("Aborting remaining pilot runs. Inspect results/generated/ and results/raw/ for the failure.")
        raise
    run([sys.executable, "scripts/run_experiment.py", "--pilot", "--retry-failed"], timeout=172800)


SKIPPED: pilot already complete (64/64 OK). Use the dual-runner cell below for the main queues.


## 5. Pilot monitoring and analysis

Rerun while the pilot is active or after interruption. The runner is resumable — every completed run is checkpointed, so re-running skips what is already done.

**Resume commands:**
- `python scripts/run_experiment.py --resume --pilot` (skips completed runs)
- `python scripts/run_experiment.py --resume --retry-failed --pilot` (also re-runs failed runs)
- `python scripts/run_experiment.py --pause` (pause after current run finishes)


In [7]:
print_status()
path = ROOT / "results" / "pilot_results.csv"
if path.exists():
    run([sys.executable, "scripts/analyze.py", str(path), "--output", "results/pilot_analysis"])
    import csv
    from collections import Counter
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    total = len(rows)
    ok = sum(1 for r in rows if r.get("status") == "ok")
    failed = sum(1 for r in rows if r.get("status") == "failed")
    cost = sum(float(r.get("cost_usd") or 0) for r in rows)
    by_cfg = Counter((r.get("suite"), r.get("tool"), r.get("backbone")) for r in rows)
    print(f"\n=== PILOT SUMMARY: {total} runs ===")
    print(f"OK: {ok} | Failed: {failed} | Total cost: ${cost:.6f}")
    print("\nConfiguration breakdown:")
    for (suite, tool, backbone), count in sorted(by_cfg.items()):
        cfg_ok = sum(1 for r in rows if r.get("suite")==suite and r.get("tool")==tool and r.get("backbone")==backbone and r.get("status")=="ok")
        print(f"  {suite}/{tool}/{backbone}: {count} runs, {cfg_ok} OK")
else:
    print("SKIPPED: no pilot_results.csv yet.")

{
  "credentials": {
    "LLM_API_KEY": true,
    "LLM_BASE_URL": true,
    "LLM_API_KEY_B2": true,
    "LLM_BASE_URL_B2": true
  },
  "files": {
    "pilot_matrix.csv": {
      "rows": 80,
      "ok": 0,
      "failed": 0,
      "pending": 80,
      "cost_usd": 0.0
    },
    "pilot_results.csv": {
      "rows": 64,
      "ok": 64,
      "failed": 0,
      "pending": 0,
      "cost_usd": 8.774848
    },
    "run_matrix.csv": {
      "rows": 10150,
      "ok": 0,
      "failed": 0,
      "pending": 10150,
      "cost_usd": 0.0
    },
    "results.csv": {
      "rows": 8119,
      "ok": 7210,
      "failed": 909,
      "pending": 0,
      "cost_usd": 2592.233596
    }
  }
}
$ /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/.venv/bin/python scripts/analyze.py /home/masoyi/Teori-dalam-pengujian-perangkat-lunak/results/pilot_results.csv --output results/pilot_analysis
Analysis successfully written to results/pilot_analysis

exit=0 elapsed=1.1s

=== PILOT SUMMARY: 64 runs ===
OK: 64 | Fa

## 6. Full run status

Report-only: the main execution (10,150 runs incl. the Pynguin baseline) is complete and checkpointed. Launching of any remaining queues is handled by the runner cells in Section 6b.


In [8]:
import csv
import tomllib

pilot_path = ROOT / "results" / "pilot_results.csv"
pilot_rows = list(csv.DictReader(pilot_path.open(newline="", encoding="utf-8"))) if pilot_path.exists() else []
pilot_ok = sum(1 for r in pilot_rows if r.get("status") == "ok")

results_path = ROOT / "results" / "results.csv"
result_rows = list(csv.DictReader(results_path.open(newline="", encoding="utf-8"))) if results_path.exists() else []
spent = sum(float(r.get("cost_usd") or 0) for r in result_rows)
config = tomllib.loads((ROOT / "config" / "experiment.toml").read_text(encoding="utf-8"))["experiment"]
cost_cap = float(config["cost_cap_usd"])
launch_cap = cost_cap * float(config["stop_launch_fraction"])
print(f"Pilot: {pilot_ok}/{len(pilot_rows)} OK | LLM spent: USD {spent:.2f} / launch cap {launch_cap:.2f}")
print("Launching is handled by the dual-runner cell below (Pynguin + LLM in parallel, separate checkpoint files).")


Pilot: 64/64 OK | LLM spent: USD 2592.23 / launch cap 4000.00
Launching is handled by the dual-runner cell below (Pynguin + LLM in parallel, separate checkpoint files).


## 6b. Runner Pynguin Baseline (CPU-Only, 15 Workers)

Menjalankan baseline algoritma pencarian genetik **Pynguin** (DYNAMOSA) tanpa panggilan LLM API.
- **Gratis**: Tidak ada panggilan API LLM (CPU compute murni).
- **Maksimal Kapasitas**: 15 worker paralel (16 threads - 1 thread navigasi) untuk kecepatan penuh.
- **Checkpoint**: Menyimpan hasil di `results/pynguin_results.csv`.


In [ ]:
# Runner Pynguin Baseline (DYNAMOSA, CPU-only, gratis)
import os
import subprocess

# Gunakan kapasitas maksimal laptop: 16 threads - 1 thread navigasi = 15 workers
workers_pyg = max(1, (os.cpu_count() or 16) - 1)

cmd_pyg = [
    sys.executable, "-u", "scripts/run_experiment.py",
    "--tool", "Pynguin",
    "--results", "results/pynguin_results.csv",
    "--workers", str(workers_pyg)
]
print(f"Menjalankan Pynguin Baseline ({workers_pyg} workers, 1 thread free untuk navigasi)...\n" + "="*60)
proc_pyg = subprocess.Popen(cmd_pyg, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc_pyg.stdout:
        print(f"[PYG] {line}", end="", flush=True)
    proc_pyg.wait()
    print("\nPynguin runner selesai.")
except KeyboardInterrupt:
    print("\nDIHENTIKAN — menghentikan runner Pynguin...")
    proc_pyg.kill()


## 6c. Runner B2 / GLM 5.3 Flash (1 Worker, Max 50 RPM)

Menjalankan benchmark LLM backbone **B2** (`glm-5.3-flash` pada `https://router.octalabs.id/v1`).
- **Model**: `glm-5.3-flash` ( upstream: `zai-org/GLM-5.3-Flash` ).
- **Strict Rate Limiting**: Maksimal 50 requests per menit (RPM) melalui `src/rate_gate.py` (inter-call delay: 1.2 detik).
- **Single In-Flight Wait**: 1 worker serial; setiap request menunggu respon provider selesai sebelum mengirim request berikutnya.
- **No Cache Hits**: Header `Cache-Control: no-cache, no-store, must-revalidate` dan prompt nonce aktif untuk mencegah proxy/prompt caching.
- **Checkpoint**: Menyimpan hasil di `results/b2_results.csv`.


In [ ]:
# Runner B2: GLM 5.3 Flash (1 worker serial, max 50 RPM gate, synchronous wait)
import subprocess

cmd_b2 = [
    sys.executable, "-u", "scripts/run_experiment.py",
    "--matrix", "results/run_matrix_b2.csv",
    "--results", "results/b2_results.csv",
    "--backbone", "B2",
    "--retry-failed",
    "--workers", "3"
]
print("Menjalankan B2 GLM 5.3 Flash (1 worker, max 50 RPM gate, single in-flight wait)...\n" + "="*60)
proc_b2 = subprocess.Popen(cmd_b2, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc_b2.stdout:
        print(f"[B2-GLM53] {line}", end="", flush=True)
    proc_b2.wait()
    print("\nB2 runner selesai.")
except KeyboardInterrupt:
    print("\nDIHENTIKAN — menghentikan runner B2...")
    proc_b2.kill()


## 6d. Runner MuTAP Suite (Opsional)

Menjalankan task khusus benchmark MuTAP jika diperlukan evaluasi ulang secara mandiri.


In [ ]:
# Runner MuTAP (Opsional jika menjalankan matrix MuTAP standalone)
from pathlib import Path
import subprocess

matrix_mutap = ROOT / "results" / "run_matrix_mutap.csv"
if not matrix_mutap.exists():
    print("results/run_matrix_mutap.csv tidak ada; data MuTAP sudah ada di results/mutap_results.csv.")
else:
    cmd_mutap = [
        sys.executable, "-u", "scripts/run_experiment.py",
        "--matrix", str(matrix_mutap),
        "--results", "results/mutap_results.csv",
        "--retry-failed",
        "--workers", "1"
    ]
    proc_mutap = subprocess.Popen(cmd_mutap, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc_mutap.stdout:
            print(f"[MuTAP] {line}", end="", flush=True)
        proc_mutap.wait()
    except KeyboardInterrupt:
        proc_mutap.kill()


## 7. Final Analysis and Research Questions (RQ1, RQ2, RQ3)

Menganalisis hasil eksperimen dan mengevaluasi 3 pertanyaan riset (RQs):
- **RQ1 (Tool Effectiveness)**: Line coverage (`lc`), branch coverage (`bc`), mutation score (`ms`), dan test validity (`rv` %). Membandingkan 4 tools pada MT suite (`MuTAP`, `Pynguin`, `CoverUp`, `CodaMOSA`) dan 3 tools pada CM suite (`Pynguin`, `CoverUp`, `CodaMOSA`) untuk setiap backbone.
- **RQ2 (Algorithms and Backbones)**: Dekomposisi varians & Linear Mixed-Effects Model (`metric ~ tool * backbone + (1 | unit)`) untuk mengisolasi efek Algoritma, Backbone, dan Interaksi keduanya.
- **RQ3 (Cost-Effectiveness)**: Biaya token (in/out), biaya moneter (USD), waktu generasi (`gen_time_s`), rasio Cost-Effectiveness (\Delta E / Cost), dan Pareto frontier.


In [ ]:
import csv
import pandas as pd

results = ROOT / "results" / "results.csv"
if not results.exists():
    print("SKIPPED: no final results.csv yet.")
else:
    with results.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    n_ok = sum(1 for r in rows if r.get("status") == "ok")
    if n_ok == 0:
        print(f"SKIPPED: results.csv has {len(rows)} rows but 0 with status=ok.")
    else:
        # 1. Jalankan analisis statistik dan pembuatan grafik
        run([sys.executable, "scripts/analyze.py", str(results)])
        run([sys.executable, "scripts/figures.py", str(results)])
        
        # 2. Tampilkan Hasil RQ1: Tool Effectiveness & Test Validity
        print("\n" + "="*70)
        print("RQ1: TOOL EFFECTIVENESS & TEST VALIDITY (Tabel IV)")
        print("="*70)
        tbl_iv = pd.read_csv(ROOT / "results" / "analysis" / "table_iv_summary.csv", header=[0, 1], index_col=[0, 1, 2])
        print(tbl_iv[[('rv', 'median'), ('lc', 'median'), ('bc', 'median'), ('ms', 'median')]])
        
        # 3. Tampilkan Hasil RQ2: Algorithms vs Backbones Variance Decomposition
        print("\n" + "="*70)
        print("RQ2: ALGORITHMS & BACKBONES (Tabel VII: Mixed Effects & Var %)")
        print("="*70)
        tbl_vii = pd.read_csv(ROOT / "results" / "analysis" / "table_vii_mixed_effects.csv")
        print(tbl_vii[['suite', 'metric', 'factor', 'estimate', 'p_value', 'var_pct']].to_string(index=False))
        
        # 4. Tampilkan Hasil RQ3: Cost-Effectiveness
        print("\n" + "="*70)
        print("RQ3: COST-EFFECTIVENESS (Tabel VIII: Cost & CE Ratio)")
        print("="*70)
        tbl_ce = pd.read_csv(ROOT / "results" / "analysis" / "table_viii_cost_effectiveness.csv")
        print(tbl_ce[['suite', 'tool', 'backbone', 'delta_e', 'median_cost_usd', 'CE']].to_string(index=False))
